# E-commerce SQL + dbt Analytics — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

Olist Brazilian E-commerce public dataset (pinned/reproducible download in src/data.py).

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'ecommerce_sql_analytics'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
"""Build and verify the Olist DuckDB analytics project."""
from __future__ import annotations

import argparse
import hashlib
import json
from dataclasses import asdict
from pathlib import Path

import duckdb
import pandas as pd

from src.config import ProjectConfig
from src.data import ensure_dataset
from src.validate import assert_integrity, run_integrity_checks
from src.warehouse import build_analytics, connect, export_analytics, load_raw_tables

RETAINED_EVIDENCE = {
    "commercial_orders": 98199,
    "unique_customers": 94983,
    "merchandise_value_brl": 13494400.74,
    "repeat_customer_pct": 3.03,
    "strongest_complete_month": "2017-11-01",
}


def _create_table(connection: duckdb.DuckDBPyConnection, name: str, frame: pd.DataFrame) -> None:
    registration = f"fixture_{name}"
    connection.register(registration, frame)
    connection.execute(f"CREATE OR REPLACE TABLE raw.{name} AS SELECT * FROM {registration}")
    connection.unregister(registration)


def build_synthetic_fixture(connection: duckdb.DuckDBPyConnection) -> None:
    """Create a tiny relational fixture that includes repeat customers and join traps."""
    customers = pd.DataFrame(
        {
            "customer_id": ["c1", "c2", "c3", "c4"],
            "customer_unique_id": ["u1", "u1", "u2", "u3"],
            "customer_zip_code_prefix": [1000, 1000, 2000, 3000],
            "customer_city": ["sao_paulo", "sao_paulo", "rio", "curitiba"],
            "customer_state": ["SP", "SP", "RJ", "PR"],
        }
    )
    orders = pd.DataFrame(
        {
            "order_id": ["o1", "o2", "o3", "o4"],
            "customer_id": ["c1", "c2", "c3", "c4"],
            "order_status": ["delivered", "delivered", "delivered", "canceled"],
            "order_purchase_timestamp": [
                "2017-01-05 08:00:00",
                "2017-02-05 09:00:00",
                "2017-02-10 10:00:00",
                "2017-03-01 11:00:00",
            ],
            "order_approved_at": [
                "2017-01-05 09:00:00",
                "2017-02-05 10:00:00",
                "2017-02-10 11:00:00",
                "2017-03-01 12:00:00",
            ],
            "order_delivered_carrier_date": [
                "2017-01-06 12:00:00",
                "2017-02-06 12:00:00",
                "2017-02-11 12:00:00",
                None,
            ],
            "order_delivered_customer_date": [
                "2017-01-10 12:00:00",
                "2017-02-20 12:00:00",
                "2017-02-14 12:00:00",
                None,
            ],
            "order_estimated_delivery_date": [
                "2017-01-12 00:00:00",
                "2017-02-15 00:00:00",
                "2017-02-16 00:00:00",
                "2017-03-20 00:00:00",
            ],
        }
    )
    order_items = pd.DataFrame(
        {
            "order_id": ["o1", "o1", "o2", "o3", "o4"],
            "order_item_id": [1, 2, 1, 1, 1],
            "product_id": ["p1", "p2", "p1", "p2", "p1"],
            "seller_id": ["s1", "s1", "s1", "s2", "s1"],
            "shipping_limit_date": ["2017-01-07"] * 5,
            "price": [100.0, 50.0, 80.0, 20.0, 999.0],
            "freight_value": [10.0, 5.0, 5.0, 5.0, 20.0],
        }
    )
    payments = pd.DataFrame(
        {
            "order_id": ["o1", "o1", "o2", "o3", "o4"],
            "payment_sequential": [1, 2, 1, 1, 1],
            "payment_type": ["credit_card", "voucher", "credit_card", "debit_card", "credit_card"],
            "payment_installments": [2, 1, 1, 1, 10],
            "payment_value": [120.0, 45.0, 85.0, 25.0, 1019.0],
        }
    )
    reviews = pd.DataFrame(
        {
            "review_id": ["r1", "r1b", "r2", "r3"],
            "order_id": ["o1", "o1", "o2", "o3"],
            "review_score": [5, 4, 2, 5],
            "review_comment_title": [None] * 4,
            "review_comment_message": [None] * 4,
            "review_creation_date": ["2017-01-11", "2017-01-12", "2017-02-21", "2017-02-15"],
            "review_answer_timestamp": [
                "2017-01-11 09:00:00",
                "2017-01-12 09:00:00",
                "2017-02-21 09:00:00",
                "2017-02-15 09:00:00",
            ],
        }
    )
    products = pd.DataFrame(
        {
            "product_id": ["p1", "p2"],
            "product_category_name": ["cat_a", "cat_b"],
            "product_name_lenght": [10, 10],
            "product_description_lenght": [20, 20],
            "product_photos_qty": [1, 1],
            "product_weight_g": [100, 200],
            "product_length_cm": [10, 20],
            "product_height_cm": [5, 5],
            "product_width_cm": [5, 10],
        }
    )
    sellers = pd.DataFrame(
        {
            "seller_id": ["s1", "s2"],
            "seller_zip_code_prefix": [1000, 2000],
            "seller_city": ["sao_paulo", "rio"],
            "seller_state": ["SP", "RJ"],
        }
    )
    category_translation = pd.DataFrame(
        {
            "product_category_name": ["cat_a", "cat_b"],
            "product_category_name_english": ["category_a", "category_b"],
        }
    )

    for name, frame in {
        "customers": customers,
        "orders": orders,
        "order_items": order_items,
        "payments": payments,
        "reviews": reviews,
        "products": products,
        "sellers": sellers,
        "category_translation": category_translation,
    }.items():
        _create_table(connection, name, frame)


def self_test() -> None:
    connection = connect(":memory:")
    build_synthetic_fixture(connection)
    build_analytics(connection, Path(__file__).parent / "sql")
    checks = run_integrity_checks(connection)
    assert_integrity(checks)

    headline = connection.execute("SELECT * FROM analytics.headline_kpis").fetchone()
    columns = [item[0] for item in connection.description]
    headline_dict = dict(zip(columns, headline))
    assert headline_dict["commercial_orders"] == 3
    assert headline_dict["unique_customers"] == 2
    assert abs(float(headline_dict["merchandise_value_brl"]) - 250.0) < 0.01

    # o1 has two items and two payment rows. A raw three-way join would create four
    # combinations; the order mart must still contain one o1 row with R$150 GMV.
    o1 = connection.execute(
        "SELECT item_count, merchandise_value_brl, payment_rows FROM analytics.order_mart WHERE order_id='o1'"
    ).fetchone()
    assert o1 == (2, 150.0, 2)

    repeat = connection.execute("SELECT repeat_customer_pct FROM analytics.customer_order_frequency").fetchone()[0]
    assert abs(float(repeat) - 50.0) < 0.01
    print("E-commerce SQL analytics self-test passed.")


def _file_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def full_run(config: ProjectConfig) -> dict[str, object]:
    source_hashes = ensure_dataset(config)
    connection = connect(config.database_path)
    load_raw_tables(connection, config.data_dir)
    build_analytics(connection, Path(__file__).parent / "sql")

    checks = run_integrity_checks(connection)
    assert_integrity(checks)
    exports = export_analytics(connection, config.output_dir / "tables")

    headline = connection.execute("SELECT * FROM analytics.headline_kpis").df().iloc[0].to_dict()
    repeat = connection.execute("SELECT * FROM analytics.customer_order_frequency").df().iloc[0].to_dict()
    strongest = connection.execute(
        "SELECT order_month, merchandise_value_brl FROM analytics.monthly_performance ORDER BY merchandise_value_brl DESC LIMIT 1"
    ).fetchone()

    observed = {
        "commercial_orders": int(headline["commercial_orders"]),
        "unique_customers": int(headline["unique_customers"]),
        "merchandise_value_brl": round(float(headline["merchandise_value_brl"]), 2),
        "repeat_customer_pct": round(float(repeat["repeat_customer_pct"]), 2),
        "strongest_complete_month": str(strongest[0]),
        "strongest_month_merchandise_value_brl": round(float(strongest[1]), 2),
    }
    retained_match = (
        observed["commercial_orders"] == RETAINED_EVIDENCE["commercial_orders"]
        and observed["unique_customers"] == RETAINED_EVIDENCE["unique_customers"]
        and abs(observed["merchandise_value_brl"] - RETAINED_EVIDENCE["merchandise_value_brl"]) < 0.01
        and abs(observed["repeat_customer_pct"] - RETAINED_EVIDENCE["repeat_customer_pct"]) < 0.01
        and observed["strongest_complete_month"].startswith(RETAINED_EVIDENCE["strongest_complete_month"])
    )

    verification = {
        "project": "E-commerce Sales and Customer Analysis",
        "verification_pass": bool(retained_match and all(check.passed for check in checks)),
        "configuration": {key: str(value) if isinstance(value, Path) else value for key, value in asdict(config).items()},
        "source_hashes": source_hashes,
        "observed": observed,
        "retained_reference": RETAINED_EVIDENCE,
        "retained_reference_match": retained_match,
        "integrity_checks": [asdict(check) for check in checks],
        "exports": {path.name: _file_sha256(path) for path in exports},
        "limitations": [
            "Historical anonymised marketplace data; results do not describe Olist's current business.",
            "Merchandise value is not profit because product cost and operating expense are unavailable.",
            "Delivery/review relationships are observational and should not be interpreted as causal effects.",
            "Later acquisition cohorts have less time to mature and are therefore right-censored.",
        ],
    }
    config.output_dir.mkdir(parents=True, exist_ok=True)
    (config.output_dir / "verification.json").write_text(json.dumps(verification, indent=2, default=str), encoding="utf-8")
    if not verification["verification_pass"]:
        raise AssertionError("Full dataset output did not match retained verified evidence")
    return verification


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Build the Olist e-commerce DuckDB analytics warehouse")
    parser.add_argument("--self-test", action="store_true", help="Run a fast synthetic relational test")
    parser.add_argument("--output-dir", type=Path, default=Path("artifacts"))
    parser.add_argument("--database", type=Path, default=Path("artifacts/ecommerce.duckdb"))
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if args.self_test:
        self_test()
        return
    config = ProjectConfig(output_dir=args.output_dir, database_path=args.database)
    verification = full_run(config)
    print(json.dumps(verification, indent=2, default=str))


if __name__ == "__main__":
    main()


### `src/config.py`


In [ ]:
"""Pinned source configuration for the Olist analytics project."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path


FILE_TABLES = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "payments",
    "olist_order_reviews_dataset.csv": "reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation",
}

EXPECTED_FILE_SHA256 = {
    "olist_customers_dataset.csv": "983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2",
    "olist_geolocation_dataset.csv": "b514f6fc991b9566aeba02aa5d67e2c3630f034b60a0e05aa0d082a3b66d88d6",
    "olist_order_items_dataset.csv": "0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279",
    "olist_order_payments_dataset.csv": "4f713964f2815dbbaa40b9488268c55aac3627bfce5aa96cf58d1f3616de3cc0",
    "olist_order_reviews_dataset.csv": "0dff69f6fed33a13648020198ea94d7ae12afbdd4904186c6cd904e27a3e1ccd",
    "olist_orders_dataset.csv": "8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf",
    "olist_products_dataset.csv": "3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc",
    "olist_sellers_dataset.csv": "1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e",
    "product_category_name_translation.csv": "a81f0d1f27b27e7293f761bc79e3ce8f348ee39c4b3ed3e49bde38f478586278",
}


@dataclass(frozen=True)
class ProjectConfig:
    dataset_url: str = (
        "https://www.kaggle.com/api/v1/datasets/download/"
        "olistbr/brazilian-ecommerce?datasetVersionNumber=7"
    )
    dataset_version: int = 7
    archive_sha256: str = "d521eb1d4a8b6dae030aa429380787261d3b04cd95bee0f43f18cb9cb18ffebb"
    data_dir: Path = Path("data/olist_v7")
    archive_path: Path = Path("data/olist_brazilian_ecommerce_v7.zip")
    database_path: Path = Path("artifacts/ecommerce.duckdb")
    output_dir: Path = Path("artifacts")
    complete_month_start: str = "2017-01-01"
    complete_month_end: str = "2018-09-01"

    def validate(self) -> None:
        if self.dataset_version != 7:
            raise ValueError("This project is verified against Olist dataset version 7")
        if self.complete_month_start >= self.complete_month_end:
            raise ValueError("complete_month_start must be before complete_month_end")


### `src/data.py`


In [ ]:
"""Download, fingerprint and extract the pinned Olist dataset."""
from __future__ import annotations

import hashlib
import shutil
import urllib.request
import zipfile
from pathlib import Path

from .config import EXPECTED_FILE_SHA256, FILE_TABLES, ProjectConfig


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def _download(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    with urllib.request.urlopen(url, timeout=120) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    temporary.replace(destination)


def verify_archive(config: ProjectConfig) -> None:
    actual = sha256_file(config.archive_path)
    if actual != config.archive_sha256:
        raise ValueError(
            "Downloaded archive hash does not match the retained dataset-v7 fingerprint: "
            f"expected {config.archive_sha256}, got {actual}"
        )


def extract_and_verify(config: ProjectConfig) -> dict[str, str]:
    """Extract the expected CSVs and fail if any file differs from the verified source."""
    config.data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(config.archive_path) as archive:
        archive_names = {Path(name).name: name for name in archive.namelist() if name.lower().endswith(".csv")}
        missing = sorted(set(FILE_TABLES) - set(archive_names))
        if missing:
            raise ValueError(f"Archive is missing expected CSV files: {missing}")
        for filename in FILE_TABLES:
            destination = config.data_dir / filename
            with archive.open(archive_names[filename]) as source, destination.open("wb") as output:
                shutil.copyfileobj(source, output)

    hashes: dict[str, str] = {}
    for filename, expected in EXPECTED_FILE_SHA256.items():
        path = config.data_dir / filename
        actual = sha256_file(path)
        hashes[filename] = actual
        if actual != expected:
            raise ValueError(f"Source fingerprint mismatch for {filename}: expected {expected}, got {actual}")
    return hashes


def ensure_dataset(config: ProjectConfig) -> dict[str, str]:
    """Make the verified dataset available locally, downloading it only when required."""
    config.validate()
    if not config.archive_path.exists():
        print("Downloading pinned Olist dataset version 7...")
        _download(config.dataset_url, config.archive_path)
    verify_archive(config)

    all_extracted = all((config.data_dir / filename).exists() for filename in FILE_TABLES)
    if all_extracted:
        hashes = {filename: sha256_file(config.data_dir / filename) for filename in FILE_TABLES}
        if hashes == EXPECTED_FILE_SHA256:
            return hashes
    return extract_and_verify(config)


### `src/warehouse.py`


In [ ]:
"""DuckDB warehouse construction and SQL execution."""
from __future__ import annotations

from pathlib import Path

import duckdb
import pandas as pd

from .config import FILE_TABLES


def connect(database_path: Path | str = ":memory:") -> duckdb.DuckDBPyConnection:
    if str(database_path) != ":memory:":
        Path(database_path).parent.mkdir(parents=True, exist_ok=True)
    connection = duckdb.connect(str(database_path))
    connection.execute("CREATE SCHEMA IF NOT EXISTS raw")
    connection.execute("CREATE SCHEMA IF NOT EXISTS analytics")
    return connection


def load_raw_tables(connection: duckdb.DuckDBPyConnection, data_dir: Path) -> None:
    """Load each verified CSV into a raw DuckDB table."""
    for filename, table_name in FILE_TABLES.items():
        path = (data_dir / filename).resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        safe_path = str(path).replace("'", "''")
        connection.execute(
            f"""
            CREATE OR REPLACE TABLE raw.{table_name} AS
            SELECT *
            FROM read_csv_auto('{safe_path}', header = TRUE, sample_size = -1, all_varchar = FALSE)
            """
        )


def execute_sql_file(connection: duckdb.DuckDBPyConnection, path: Path) -> None:
    sql = path.read_text(encoding="utf-8")
    connection.execute(sql)


def build_analytics(connection: duckdb.DuckDBPyConnection, sql_dir: Path) -> None:
    """Execute numbered SQL modules in deterministic order."""
    sql_files = sorted(sql_dir.glob("*.sql"))
    if not sql_files:
        raise FileNotFoundError(f"No SQL files found under {sql_dir}")
    for path in sql_files:
        print(f"Executing {path.name}")
        execute_sql_file(connection, path)


def dataframe(connection: duckdb.DuckDBPyConnection, query: str) -> pd.DataFrame:
    return connection.execute(query).df()


def export_analytics(connection: duckdb.DuckDBPyConnection, output_dir: Path) -> list[Path]:
    """Export compact recruiter-readable result tables as Parquet."""
    output_dir.mkdir(parents=True, exist_ok=True)
    tables = [
        "headline_kpis",
        "monthly_performance",
        "customer_order_frequency",
        "cohort_retention",
        "category_performance",
        "delivery_review_summary",
        "seller_operational_review",
        "seller_concentration_summary",
        "payment_behaviour",
        "top_categories_by_customer_state",
    ]
    outputs: list[Path] = []
    for table in tables:
        destination = (output_dir / f"{table}.parquet").resolve()
        safe_destination = str(destination).replace("'", "''")
        connection.execute(
            f"COPY analytics.{table} TO '{safe_destination}' (FORMAT PARQUET, COMPRESSION ZSTD)"
        )
        outputs.append(destination)
    return outputs


### `src/validate.py`


In [ ]:
"""Warehouse integrity, grain and financial reconciliation checks."""
from __future__ import annotations

from dataclasses import dataclass

import duckdb


@dataclass(frozen=True)
class CheckResult:
    name: str
    passed: bool
    value: float | int
    expectation: str


def _scalar(connection: duckdb.DuckDBPyConnection, query: str) -> float | int:
    value = connection.execute(query).fetchone()[0]
    return 0 if value is None else value


def run_integrity_checks(connection: duckdb.DuckDBPyConnection) -> list[CheckResult]:
    """Check keys, foreign keys, semantic grain and headline financial reconciliation."""
    checks: list[CheckResult] = []

    uniqueness_checks = {
        "customers_customer_id_unique": ("raw.customers", "customer_id"),
        "orders_order_id_unique": ("raw.orders", "order_id"),
        "products_product_id_unique": ("raw.products", "product_id"),
        "sellers_seller_id_unique": ("raw.sellers", "seller_id"),
    }
    for name, (table, key) in uniqueness_checks.items():
        duplicates = int(
            _scalar(
                connection,
                f"SELECT COUNT(*) FROM (SELECT {key} FROM {table} GROUP BY {key} HAVING COUNT(*) > 1)",
            )
        )
        checks.append(CheckResult(name, duplicates == 0, duplicates, "0 duplicate keys"))

    orphan_queries = {
        "orders_customer_fk": """
            SELECT COUNT(*) FROM raw.orders o
            LEFT JOIN raw.customers c USING (customer_id)
            WHERE c.customer_id IS NULL
        """,
        "items_order_fk": """
            SELECT COUNT(*) FROM raw.order_items i
            LEFT JOIN raw.orders o USING (order_id)
            WHERE o.order_id IS NULL
        """,
        "items_product_fk": """
            SELECT COUNT(*) FROM raw.order_items i
            LEFT JOIN raw.products p USING (product_id)
            WHERE p.product_id IS NULL
        """,
        "items_seller_fk": """
            SELECT COUNT(*) FROM raw.order_items i
            LEFT JOIN raw.sellers s USING (seller_id)
            WHERE s.seller_id IS NULL
        """,
        "payments_order_fk": """
            SELECT COUNT(*) FROM raw.payments p
            LEFT JOIN raw.orders o USING (order_id)
            WHERE o.order_id IS NULL
        """,
        "reviews_order_fk": """
            SELECT COUNT(*) FROM raw.reviews r
            LEFT JOIN raw.orders o USING (order_id)
            WHERE o.order_id IS NULL
        """,
    }
    for name, query in orphan_queries.items():
        orphans = int(_scalar(connection, query))
        checks.append(CheckResult(name, orphans == 0, orphans, "0 orphan rows"))

    raw_orders = int(_scalar(connection, "SELECT COUNT(*) FROM raw.orders"))
    mart_orders = int(_scalar(connection, "SELECT COUNT(*) FROM analytics.order_mart"))
    distinct_mart_orders = int(_scalar(connection, "SELECT COUNT(DISTINCT order_id) FROM analytics.order_mart"))
    checks.extend(
        [
            CheckResult("order_mart_row_count", mart_orders == raw_orders, mart_orders, f"{raw_orders} rows"),
            CheckResult(
                "order_mart_one_row_per_order",
                distinct_mart_orders == mart_orders,
                distinct_mart_orders,
                f"{mart_orders} distinct order ids",
            ),
        ]
    )

    raw_items = int(_scalar(connection, "SELECT COUNT(*) FROM raw.order_items"))
    mart_items = int(_scalar(connection, "SELECT COUNT(*) FROM analytics.item_mart"))
    checks.append(CheckResult("item_mart_row_count", mart_items == raw_items, mart_items, f"{raw_items} rows"))

    order_value = float(
        _scalar(
            connection,
            "SELECT COALESCE(SUM(merchandise_value_brl), 0) FROM analytics.order_mart WHERE commercial_order",
        )
    )
    item_value = float(
        _scalar(
            connection,
            "SELECT COALESCE(SUM(item_price_brl), 0) FROM analytics.item_mart WHERE commercial_order",
        )
    )
    delta = abs(order_value - item_value)
    checks.append(CheckResult("merchandise_value_reconciliation", delta < 0.01, round(delta, 6), "< R$0.01 difference"))

    negative_prices = int(
        _scalar(connection, "SELECT COUNT(*) FROM raw.order_items WHERE price < 0 OR freight_value < 0")
    )
    checks.append(CheckResult("non_negative_item_values", negative_prices == 0, negative_prices, "0 negative values"))

    invalid_reviews = int(
        _scalar(connection, "SELECT COUNT(*) FROM raw.reviews WHERE review_score NOT BETWEEN 1 AND 5")
    )
    checks.append(CheckResult("review_score_range", invalid_reviews == 0, invalid_reviews, "scores between 1 and 5"))

    return checks


def assert_integrity(checks: list[CheckResult]) -> None:
    failed = [check for check in checks if not check.passed]
    if failed:
        summary = "; ".join(f"{check.name}={check.value} expected {check.expectation}" for check in failed)
        raise AssertionError(f"Warehouse integrity checks failed: {summary}")


### `dbt_project/prepare_fixture.py`


In [ ]:
"""Create a deterministic DuckDB source fixture for the dbt smoke test."""
from pathlib import Path

import duckdb

DB_PATH = Path(__file__).parent / "artifacts" / "dbt_smoke.duckdb"


def main() -> None:
    DB_PATH.parent.mkdir(parents=True, exist_ok=True)
    if DB_PATH.exists():
        DB_PATH.unlink()

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE SCHEMA raw")

    con.execute("""
        CREATE TABLE raw.customers (
            customer_id VARCHAR,
            customer_unique_id VARCHAR,
            customer_city VARCHAR,
            customer_state VARCHAR
        )
    """)
    con.execute("""
        INSERT INTO raw.customers VALUES
        ('c1','u1','sao_paulo','SP'),
        ('c2','u1','sao_paulo','SP'),
        ('c3','u2','rio','RJ')
    """)

    con.execute("""
        CREATE TABLE raw.orders (
            order_id VARCHAR,
            customer_id VARCHAR,
            order_status VARCHAR,
            order_purchase_timestamp VARCHAR,
            order_delivered_customer_date VARCHAR,
            order_estimated_delivery_date VARCHAR
        )
    """)
    con.execute("""
        INSERT INTO raw.orders VALUES
        ('o1','c1','delivered','2017-01-05 08:00:00','2017-01-10 12:00:00','2017-01-12 00:00:00'),
        ('o2','c2','delivered','2017-02-05 09:00:00','2017-02-20 12:00:00','2017-02-15 00:00:00'),
        ('o3','c3','delivered','2017-02-10 10:00:00','2017-02-14 12:00:00','2017-02-16 00:00:00')
    """)

    con.execute("""
        CREATE TABLE raw.order_items (
            order_id VARCHAR,
            order_item_id INTEGER,
            product_id VARCHAR,
            seller_id VARCHAR,
            price DOUBLE,
            freight_value DOUBLE
        )
    """)
    con.execute("""
        INSERT INTO raw.order_items VALUES
        ('o1',1,'p1','s1',100.0,10.0),
        ('o1',2,'p2','s1',50.0,5.0),
        ('o2',1,'p1','s1',80.0,5.0),
        ('o3',1,'p2','s2',20.0,5.0)
    """)

    con.execute("""
        CREATE TABLE raw.payments (
            order_id VARCHAR,
            payment_sequential INTEGER,
            payment_type VARCHAR,
            payment_installments INTEGER,
            payment_value DOUBLE
        )
    """)
    con.execute("""
        INSERT INTO raw.payments VALUES
        ('o1',1,'credit_card',2,120.0),
        ('o1',2,'voucher',1,45.0),
        ('o2',1,'credit_card',1,85.0),
        ('o3',1,'debit_card',1,25.0)
    """)

    con.execute("""
        CREATE TABLE raw.reviews (
            review_id VARCHAR,
            order_id VARCHAR,
            review_score INTEGER,
            review_creation_date VARCHAR,
            review_answer_timestamp VARCHAR
        )
    """)
    con.execute("""
        INSERT INTO raw.reviews VALUES
        ('r1','o1',5,'2017-01-11','2017-01-11 09:00:00'),
        ('r1b','o1',4,'2017-01-12','2017-01-12 09:00:00'),
        ('r2','o2',2,'2017-02-21','2017-02-21 09:00:00'),
        ('r3','o3',5,'2017-02-15','2017-02-15 09:00:00')
    """)

    con.close()
    print(f"Prepared dbt fixture: {DB_PATH}")


if __name__ == "__main__":
    main()


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


## Application engineering layer

The original project above is intentionally preserved. The cells below expose additional canonical Python from this same project—pipelines, APIs, feature code, evaluation, tests, monitoring and other application logic—so the notebook works as a single recruiter-facing project while the modular files remain the production source of truth.


### Canonical source: `src/__init__.py`


In [ ]:
"""E-commerce SQL analytics project package."""


### Canonical source: `tests/test_sql_logic.py`


In [ ]:
from __future__ import annotations

import unittest
from pathlib import Path

from run import build_synthetic_fixture
from src.validate import assert_integrity, run_integrity_checks
from src.warehouse import build_analytics, connect


class EcommerceSqlTests(unittest.TestCase):
    def setUp(self) -> None:
        self.connection = connect(":memory:")
        build_synthetic_fixture(self.connection)
        build_analytics(self.connection, Path(__file__).parents[1] / "sql")

    def tearDown(self) -> None:
        self.connection.close()

    def test_integrity_checks_pass(self) -> None:
        checks = run_integrity_checks(self.connection)
        assert_integrity(checks)
        self.assertTrue(all(check.passed for check in checks))

    def test_semantic_mart_prevents_many_to_many_revenue_inflation(self) -> None:
        naive_value = self.connection.execute(
            """
            SELECT SUM(i.price)
            FROM raw.order_items i
            JOIN raw.payments p USING (order_id)
            WHERE i.order_id = 'o1'
            """
        ).fetchone()[0]
        mart_value = self.connection.execute(
            "SELECT merchandise_value_brl FROM analytics.order_mart WHERE order_id = 'o1'"
        ).fetchone()[0]
        self.assertEqual(float(naive_value), 300.0)
        self.assertEqual(float(mart_value), 150.0)

    def test_latest_review_is_selected_once_per_order(self) -> None:
        score = self.connection.execute(
            "SELECT review_score FROM analytics.order_mart WHERE order_id = 'o1'"
        ).fetchone()[0]
        self.assertEqual(int(score), 4)

    def test_canceled_order_is_not_commercial(self) -> None:
        commercial = self.connection.execute(
            "SELECT commercial_order FROM analytics.order_mart WHERE order_id = 'o4'"
        ).fetchone()[0]
        self.assertFalse(bool(commercial))

    def test_commercial_scope_is_independent_of_complete_month_reporting_window(self) -> None:
        """A valid order outside the comparison window stays commercial but not in monthly KPIs."""
        self.connection.execute(
            "INSERT INTO raw.customers VALUES ('c5', 'u4', 4000, 'campinas', 'SP')"
        )
        self.connection.execute(
            """
            INSERT INTO raw.orders VALUES (
                'o5', 'c5', 'delivered',
                '2016-12-20 08:00:00', '2016-12-20 09:00:00',
                '2016-12-21 12:00:00', '2016-12-28 12:00:00', '2016-12-30 00:00:00'
            )
            """
        )
        self.connection.execute(
            "INSERT INTO raw.order_items VALUES ('o5', 1, 'p1', 's1', '2016-12-22', 40.0, 5.0)"
        )
        build_analytics(self.connection, Path(__file__).parents[1] / "sql")

        commercial = self.connection.execute(
            "SELECT commercial_order FROM analytics.order_mart WHERE order_id = 'o5'"
        ).fetchone()[0]
        headline_orders = self.connection.execute(
            "SELECT commercial_orders FROM analytics.headline_kpis"
        ).fetchone()[0]
        out_of_window_months = self.connection.execute(
            "SELECT COUNT(*) FROM analytics.monthly_performance WHERE order_month < DATE '2017-01-01'"
        ).fetchone()[0]

        self.assertTrue(bool(commercial))
        self.assertEqual(int(headline_orders), 4)
        self.assertEqual(int(out_of_window_months), 0)

    def test_repeat_customer_cohort_is_preserved(self) -> None:
        month_one = self.connection.execute(
            """
            SELECT active_customers, cohort_customers, retention_pct
            FROM analytics.cohort_retention
            WHERE cohort_month = DATE '2017-01-01' AND month_number = 1
            """
        ).fetchone()
        self.assertEqual(tuple(month_one), (1, 1, 100.0))

    def test_qualify_returns_at_most_three_categories_per_state(self) -> None:
        max_categories = self.connection.execute(
            """
            SELECT MAX(category_count)
            FROM (
                SELECT customer_state, COUNT(*) AS category_count
                FROM analytics.top_categories_by_customer_state
                GROUP BY customer_state
            )
            """
        ).fetchone()[0]
        self.assertLessEqual(int(max_categories), 3)


if __name__ == "__main__":
    unittest.main()


## Portfolio depth check

**Meaningful code lines currently visible in this notebook:** 726.  The portfolio aims for roughly **1,000 meaningful lines** per major project (normally about 800–1,200), and this notebook is below the preferred band and should gain substantive project-specific depth rather than filler.

Line count is not a quality metric by itself. Additional code should only be added when it strengthens the real application: data validation, cleaning, EDA, feature engineering, modelling, tuning, error analysis, explainability, inference, testing, monitoring, APIs, reproducibility or business logic.
